In [1]:
import numpy as np, sys
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split

# 1. Set the random seed to 42 as required by the assignment
np.random.seed(42)

# Load the data and merge it to create a custom split
(X_train_orig, y_train_orig), (X_test_orig, y_test_orig) = mnist.load_data()
X_all = np.concatenate((X_train_orig, X_test_orig), axis=0)
y_all = np.concatenate((y_train_orig, y_test_orig), axis=0)

# 2. Apply test_size=0.2 and random_state=42 as requested
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

# Limit to first 1000 for faster training and testing
X_train = X_train[:1000]
labels = y_train[:1000]
X_test = X_test[:1000]
y_test = y_test[:1000]

# Preprocess training data
images = X_train.reshape(len(X_train), 28*28) / 255

# One-hot encode training labels
one_hot_labels = np.zeros((len(labels), 10))
for i, l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

# Preprocess testing data
test_images = X_test.reshape(len(X_test), 28*28) / 255

# One-hot encode test labels
test_labels = np.zeros((len(y_test), 10))
for i, l in enumerate(y_test):
    test_labels[i][l] = 1

# Hyperparameters
alpha, iterations = (0.1, 300)
pixels_per_image, num_labels = (784, 10)
batch_size = 128

input_rows = 28
input_cols = 28

# 3. Use kernel of size 3x3 and 16 of them
kernel_rows = 3
kernel_cols = 3
num_kernels = 16

hidden_size = ((input_rows - kernel_rows) * (input_cols - kernel_cols)) * num_kernels

# Initialize Weights and Kernels
kernels = np.random.random((kernel_rows * kernel_cols, num_kernels))
weights_1_2 = np.random.random((hidden_size, num_labels))

def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

# --- Training Loop ---
for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i+1) * batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape[0], 28, 28)
        sects = list()
        for row_start in range(layer_0.shape[1] - kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0, row_start, row_start + kernel_rows, col_start, col_start + kernel_cols)
                sects.append(sect)
        expanded_input = np.concatenate(sects, axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0] * es[1], -1)
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0], -1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1, weights_1_2))
        correct_cnt += np.sum(np.argmax(layer_2, axis=1) == np.argmax(labels[batch_start:batch_end], axis=1))
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / batch_size
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        k_update = flattened_input.T.dot(l1d_reshape)
        kernels -= alpha * k_update
    test_correct_cnt = 0
    for i in range(len(test_images)):
        layer_0 = test_images[i:i+1]
        layer_0 = layer_0.reshape(layer_0.shape[0], 28, 28)
        sects = list()
        for row_start in range(layer_0.shape[1] - kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0, row_start, row_start + kernel_rows, col_start, col_start + kernel_cols)
                sects.append(sect)
        expanded_input = np.concatenate(sects, axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0] * es[1], -1)
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0], -1))
        layer_2 = np.dot(layer_1, weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

    if (j % 10 == 0) or (j == iterations - 1):
        sys.stdout.write("Iteration:" + str(j) +
                         " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) +
                         " Train-Acc:" + str(correct_cnt / float(len(images))) + "\n")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/tmp/ipython-input-3486700883.py:69: RuntimeWarning: overflow encountered in exp
  temp = np.exp(x)
/tmp/ipython-input-3486700883.py:70: RuntimeWarning: invalid value encountered in divide
  return temp / np.sum(temp, axis=1, keepdims=True)


Iteration:0 Test-Acc:0.095 Train-Acc:0.103
Iteration:10 Test-Acc:0.095 Train-Acc:0.103
Iteration:20 Test-Acc:0.095 Train-Acc:0.103


KeyboardInterrupt: 